# EFRiskEngine — EDA & Fraud Patterns

**End-to-end demo notebook** showing the full EFRiskEngine pipeline:
1. ETL (Load → Normalise → SQLite)
2. Rules-Based Risk Scoring
3. AI-Assisted Analysis (Gemini)
4. Reporting
5. Inline visualisations

---
> **Prerequisites:** Run `python generate_sample_data.py` from project root first.

In [ ]:
import sys, os
# Make sure project root is on the path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
print('✅ Imports OK')

## 1. ETL Pipeline

In [ ]:
from src.data_pipeline import run_etl

users_df, txns_df = run_etl()
print(f'Users:        {users_df.shape}')
print(f'Transactions: {txns_df.shape}')
display(users_df.head(3))
display(txns_df.head(3))

## 2. Exploratory Data Analysis

In [ ]:
print('--- Users ---')
display(users_df.describe())
print('\nFraudulent users:', users_df['is_fraudulent'].sum())
print('\n--- Transactions ---')
display(txns_df.describe())
print('\nFlagged transactions:', txns_df['is_flagged'].sum())

In [ ]:
# Purchase amount distribution
fig = px.histogram(
    txns_df, x='purchase_amount', nbins=40,
    color='is_flagged', color_discrete_map={0: '#22c55e', 1: '#ef4444'},
    title='Purchase Amount Distribution — Flagged vs Normal',
    labels={'purchase_amount': 'Amount ($)', 'is_flagged': 'Flagged'}
)
fig.show()

In [ ]:
# Country mismatch analysis
mismatch = txns_df[txns_df['transaction_country'] != txns_df['user_registered_country']]
print(f'Country mismatches: {len(mismatch)} / {len(txns_df)} ({len(mismatch)/len(txns_df)*100:.1f}%)')

fig2 = px.bar(
    mismatch['transaction_country'].value_counts().head(10).reset_index(),
    x='transaction_country', y='count',
    title='Top Transaction Countries with Country Mismatches',
    color='count', color_continuous_scale='Reds',
    labels={'transaction_country': 'Country', 'count': 'Mismatch Count'}
)
fig2.show()

In [ ]:
# Failed logins vs fraud
fig3 = px.box(
    users_df, x='is_fraudulent', y='failed_login_count',
    title='Failed Login Counts — Fraudulent vs Normal Users',
    labels={'is_fraudulent': 'Is Fraudulent (0/1)', 'failed_login_count': 'Failed Logins'},
    color='is_fraudulent', color_discrete_map={0: '#22c55e', 1: '#ef4444'}
)
fig3.show()

## 3. Rules-Based Risk Engine

In [ ]:
from src.risk_engine import run_risk_engine

scored_txns, user_summaries = run_risk_engine(txns_df, users_df)

print('Top 10 highest-risk transactions:')
display(
    scored_txns.sort_values('risk_score', ascending=False)
    .head(10)[['transaction_id','user_id','purchase_amount','risk_score','risk_label','reasons']]
)
print('\nTop 10 highest-risk users:')
display(user_summaries.head(10))

In [ ]:
# Risk score distribution
fig4 = px.histogram(
    scored_txns, x='risk_score', nbins=20,
    color='risk_label',
    color_discrete_map={'Low':'#22c55e','Medium':'#eab308','High':'#f97316','Critical':'#ef4444'},
    title='Risk Score Distribution by Label',
    category_orders={'risk_label': ['Low','Medium','High','Critical']}
)
fig4.show()

In [ ]:
# Risk over time
ts = scored_txns.copy()
ts['date'] = pd.to_datetime(ts['timestamp']).dt.date
daily = ts.groupby('date').agg(flagged=('risk_score', lambda x: (x>=25).sum()), total=('risk_score','count')).reset_index()

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=daily['date'], y=daily['total'], name='Total', line=dict(color='gray')))
fig5.add_trace(go.Scatter(x=daily['date'], y=daily['flagged'], name='Flagged (≥25)', fill='tozeroy',
                           line=dict(color='#f97316')))
fig5.update_layout(title='Daily Transactions: Total vs Flagged', hovermode='x unified')
fig5.show()

## 4. AI-Assisted Analysis

In [ ]:
from src.ai_assist import get_provider, run_ai_analysis, detect_emerging_patterns

# Use mock provider for offline demo — change to 'gemini' for live API calls
provider = get_provider('mock')
print(f'Using provider: {provider.name}')

# Analyse top 5 users
ai_results = run_ai_analysis(scored_txns, users_df, provider, top_n=5)
display(ai_results[['user_id','rules_risk_score','ai_bias_score','combined_score','combined_label']])

In [ ]:
# Emerging pattern detection across all flagged transactions
flagged = scored_txns[scored_txns['risk_score'] >= 50]
pattern_analysis = detect_emerging_patterns(flagged, provider)
print('=== Emerging Fraud Patterns ===')
print(pattern_analysis)

## 5. Reporting

In [ ]:
from src.reporting import run_reporting, summary_stats

stats = summary_stats(scored_txns, user_summaries)
print('Summary stats:')
for k, v in stats.items():
    print(f'  {k}: {v}')

# Write reports to reports/ directory
report_paths = run_reporting(scored_txns, user_summaries)
print('\nReports written:')
for name, path in report_paths.items():
    print(f'  {name}: {path}')

## 6. Next Steps

- Launch the interactive dashboard: `streamlit run dashboards/fraud_dashboard.py`
- Open `AI_analysis_unstructured_data.ipynb` for deep-dive AI prompt engineering
- Run `pytest tests/ -v` to validate all scoring logic